In [0]:
import json
import random
import time
import uuid
from datetime import datetime, timedelta

output_path = "/Volumes/server_catalog/server_schema/my_volume/raw_data/metrics"

servers = [f"srv_{i:03d}" for i in range(1,21)]

# maintain previous metrics for time-series behavior
server_state = {
    s: {
        "cpu": random.randint(20,40),
        "memory": random.randint(30,50),
        "disk": random.randint(20,40)
    }
    for s in servers
}

# servers currently degrading
failing_servers = {}

while True:

    timestamp = datetime.utcnow()
    batch = []

    for server in servers:

        prev = server_state[server]

        cpu = max(0, min(100, prev["cpu"] + random.randint(-2,4)))
        memory = max(0, min(100, prev["memory"] + random.randint(-2,3)))
        disk = max(0, min(100, prev["disk"] + random.randint(-1,2)))

        # start degradation randomly
        if random.random() < 0.05:
            failing_servers[server] = random.randint(3,6)

        if server in failing_servers:
            cpu += random.randint(5,10)
            memory += random.randint(5,10)
            disk += random.randint(3,6)

            failing_servers[server] -= 1

            if failing_servers[server] <= 0:
                del failing_servers[server]

        server_state[server] = {
            "cpu": cpu,
            "memory": memory,
            "disk": disk
        }

        record = {
            "event_id": str(uuid.uuid4()),
            "timestamp": timestamp.isoformat(),
            "server_id": server,
            "cpu": cpu,
            "memory": memory,
            "disk": disk
        }

        # null simulation
        if random.random() < 0.05:
            record["cpu"] = None

        # bad record
        if random.random() < 0.03:
            record["cpu"] = "invalid_value"

        # late arriving data
        if random.random() < 0.03:
            record["timestamp"] = (
                timestamp - timedelta(minutes=5)
            ).isoformat()

        batch.append(record)

        # duplicate record in same file
        if random.random() < 0.05:
            batch.append(record)

    file_name = f"{output_path}/metrics_{int(time.time()*1000)}.json"

    with open(file_name, "w") as f:
        json.dump(batch, f)

    print("metrics snapshot generated")

    time.sleep(3)

In [0]:
import random
import time
from datetime import datetime, timedelta

output_path = "/Volumes/server_catalog/server_schema/my_volume/raw_data/logs"

servers = [f"srv_{i:03d}" for i in range(1,21)]

levels = ["INFO", "WARN", "ERROR"]

messages = [
    "service started successfully",
    "disk error detected",
    "kernel panic occurred",
    "memory allocation failed",
    "database connection timeout",
    "network packet loss detected",
    "high cpu usage detected",
    "service restart triggered",
    "disk read latency increased",
    "connection pool exhausted"
]

while True:

    metrics_time = datetime.utcnow()
    lines = []

    # generate logs for this snapshot window
    for _ in range(random.randint(15,30)):

        server = random.choice(servers)
        level = random.choice(levels)
        message = random.choice(messages)

        log_time = metrics_time

        # logs slightly after metrics
        if random.random() < 0.6:
            log_time = metrics_time + timedelta(seconds=random.randint(1,5))

        # late arriving logs (rare)
        if random.random() < 0.03:
            log_time = metrics_time - timedelta(minutes=random.randint(1,3))

        log_line = f"{log_time.isoformat()} {server} {level} {message}"

        lines.append(log_line)

        # duplicate log
        if random.random() < 0.05:
            lines.append(log_line)

        # corrupted log entry
        if random.random() < 0.02:
            lines.append("corrupted_log_entry")

    file_name = f"{output_path}/logs_{int(time.time()*1000)}.log"

    with open(file_name, "w") as f:
        f.write("\n".join(lines))

    print("logs snapshot generated")

    time.sleep(3)

In [0]:
import csv
import random
import time
import uuid
from datetime import datetime, timedelta

output_path = "/Volumes/server_catalog/server_schema/my_volume/raw_data/events"

servers = [f"srv_{i:03d}" for i in range(1,21)]

event_types = [
    "server_crash",
    "service_restart",
    "deployment",
    "config_update",
    "scaling_event"
]

while True:

    snapshot_time = datetime.utcnow()
    rows = []

    # generate few events per snapshot
    for _ in range(random.randint(1,4)):

        server = random.choice(servers)

        event_id = str(uuid.uuid4())
        event_type = random.choice(event_types)
        timestamp = snapshot_time

        # simulate late event
        if random.random() < 0.03:
            timestamp = snapshot_time - timedelta(minutes=random.randint(1,5))

        record = [
            event_id,
            timestamp.isoformat(),
            server,
            event_type
        ]

        rows.append(record)

        # duplicate record
        if random.random() < 0.05:
            rows.append(record)

        # bad record
        if random.random() < 0.03:
            rows.append(["bad_event", "", "", ""])

    file_name = f"{output_path}/events_{int(time.time()*1000)}.csv"

    with open(file_name, "w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow([
            "event_id",
            "timestamp",
            "server_id",
            "event_type"
        ])

        writer.writerows(rows)

    print("events snapshot generated")

    time.sleep(4)